# 05 - Transfer Learning with ResNet18 (PyTorch, GPU)

This notebook trains a **ResNet18 transfer learning model** on **Dataset 1**.

## Workflow
1. Set paths and hyperparameters
2. Load train / val / test datasets
3. Use a pretrained ResNet18 backbone
4. Replace the final classification layer for 50 classes
5. Train on GPU if available
6. Save the best model and reports
7. Evaluate on the test set
8. Visualize results

## Notes
- This notebook is designed for **PyTorch + GPU**
- It keeps output file names clear and easy to distinguish
- It is suitable as the **second major model** for Assignment 2


In [ ]:
# Imports
import os
import json
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

In [ ]:
# Optional: display Chinese class names correctly on Windows
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "SimSun", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# Paths
PROJECT_ROOT = Path(r"I:\DeepLearning\ChineseHerb_Identify")

DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "herb50_dataset_1" / "split_dataset"
TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"
TEST_DIR = DATA_ROOT / "test"

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
REPORTS_DIR = RESULTS_DIR / "reports"
MODELS_DIR = PROJECT_ROOT / "models" / "transfer_learning"

for p in [FIGURES_DIR, REPORTS_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("TRAIN_DIR:", TRAIN_DIR)
print("VAL_DIR  :", VAL_DIR)
print("TEST_DIR :", TEST_DIR)

In [ ]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type != "cuda":
    print("WARNING: GPU is not being used. Check your PyTorch installation and CUDA setup.")

In [ ]:
# Hyperparameters
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 45
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0   # Safer for Windows notebook
PATIENCE = 5
FREEZE_BACKBONE = False

print("IMG_SIZE =", IMG_SIZE)
print("BATCH_SIZE =", BATCH_SIZE)
print("EPOCHS =", EPOCHS)
print("LEARNING_RATE =", LEARNING_RATE)
print("WEIGHT_DECAY =", WEIGHT_DECAY)
print("FREEZE_BACKBONE =", FREEZE_BACKBONE)

In [ ]:
# Data transforms
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
# Datasets
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=eval_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Number of classes:", num_classes)
print("First 10 classes:", class_names[:10])

In [ ]:
# Save class mapping
transfer_resnet18_class_mapping_df = pd.DataFrame({
    "class_id": list(range(len(class_names))),
    "class_name_cn": class_names
})

transfer_resnet18_class_mapping_path = REPORTS_DIR / "transfer_resnet18_pytorch_class_mapping.csv"
transfer_resnet18_class_mapping_df.to_csv(transfer_resnet18_class_mapping_path, index=False, encoding="utf-8-sig")

print("Saved:", transfer_resnet18_class_mapping_path)
transfer_resnet18_class_mapping_df.head()

In [ ]:
# Dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))

In [ ]:
# Visualize a few training images
display_images, display_labels = next(iter(train_loader))

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

plt.figure(figsize=(10, 10))
for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    img = display_images[i].cpu().permute(1, 2, 0).numpy()
    img = np.clip((img * std) + mean, 0, 1)
    plt.imshow(img)
    plt.title(class_names[display_labels[i].item()])
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Build pretrained ResNet18 model
weights = ResNet18_Weights.DEFAULT
transfer_resnet18_model = models.resnet18(weights=weights)

if FREEZE_BACKBONE:
    for param in transfer_resnet18_model.parameters():
        param.requires_grad = False

in_features = transfer_resnet18_model.fc.in_features
transfer_resnet18_model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, num_classes)
)

transfer_resnet18_model = transfer_resnet18_model.to(device)
transfer_resnet18_model

In [ ]:
# Loss, optimizer, scheduler
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, transfer_resnet18_model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

In [ ]:
# Training and validation functions
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in tqdm(loader, leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc, np.array(all_labels), np.array(all_preds)

In [ ]:
# Train loop
transfer_resnet18_history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

transfer_resnet18_best_val_acc = 0.0
transfer_resnet18_best_model_wts = copy.deepcopy(transfer_resnet18_model.state_dict())
transfer_resnet18_best_model_path = MODELS_DIR / "transfer_resnet18_pytorch_best_model.pth"

patience_counter = 0

for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1}/{EPOCHS}")

    train_loss, train_acc = train_one_epoch(
        transfer_resnet18_model, train_loader, criterion, optimizer, device
    )
    val_loss, val_acc, _, _ = evaluate(
        transfer_resnet18_model, val_loader, criterion, device
    )

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    transfer_resnet18_history["train_loss"].append(train_loss)
    transfer_resnet18_history["train_acc"].append(train_acc)
    transfer_resnet18_history["val_loss"].append(val_loss)
    transfer_resnet18_history["val_acc"].append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print(f"Learning Rate: {current_lr:.6f}")

    if val_acc > transfer_resnet18_best_val_acc:
        transfer_resnet18_best_val_acc = val_acc
        transfer_resnet18_best_model_wts = copy.deepcopy(transfer_resnet18_model.state_dict())
        torch.save(transfer_resnet18_best_model_wts, transfer_resnet18_best_model_path)
        patience_counter = 0
        print("Best model updated.")
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{PATIENCE}")

    print("-" * 60)

    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

print("Best validation accuracy:", transfer_resnet18_best_val_acc)
print("Best model saved to:", transfer_resnet18_best_model_path)

In [ ]:
# Save training history
transfer_resnet18_history_df = pd.DataFrame(transfer_resnet18_history)
transfer_resnet18_history_path = REPORTS_DIR / "transfer_resnet18_pytorch_training_history.csv"
transfer_resnet18_history_df.to_csv(transfer_resnet18_history_path, index=False)

print("Saved:", transfer_resnet18_history_path)
transfer_resnet18_history_df.head()

In [ ]:
# Plot accuracy
plt.figure(figsize=(8, 5))
plt.plot(transfer_resnet18_history["train_acc"], label="Train Accuracy")
plt.plot(transfer_resnet18_history["val_acc"], label="Val Accuracy")
plt.title("Transfer ResNet18 PyTorch Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()

transfer_resnet18_accuracy_curve_path = FIGURES_DIR / "transfer_resnet18_pytorch_accuracy_curve.png"
plt.savefig(transfer_resnet18_accuracy_curve_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", transfer_resnet18_accuracy_curve_path)

In [ ]:
# Plot loss
plt.figure(figsize=(8, 5))
plt.plot(transfer_resnet18_history["train_loss"], label="Train Loss")
plt.plot(transfer_resnet18_history["val_loss"], label="Val Loss")
plt.title("Transfer ResNet18 PyTorch Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

transfer_resnet18_loss_curve_path = FIGURES_DIR / "transfer_resnet18_pytorch_loss_curve.png"
plt.savefig(transfer_resnet18_loss_curve_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", transfer_resnet18_loss_curve_path)

In [ ]:
# Load best weights before final testing
transfer_resnet18_model.load_state_dict(torch.load(transfer_resnet18_best_model_path, map_location=device))
transfer_resnet18_model.to(device)

In [ ]:
# Test evaluation
transfer_resnet18_test_loss, transfer_resnet18_test_acc, transfer_resnet18_y_true, transfer_resnet18_y_pred = evaluate(
    transfer_resnet18_model, test_loader, criterion, device
)

print(f"Transfer ResNet18 Test Loss: {transfer_resnet18_test_loss:.4f}")
print(f"Transfer ResNet18 Test Accuracy: {transfer_resnet18_test_acc:.4f}")

In [ ]:
# Classification report
transfer_resnet18_report_dict = classification_report(
    transfer_resnet18_y_true,
    transfer_resnet18_y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

transfer_resnet18_report_df = pd.DataFrame(transfer_resnet18_report_dict).transpose()
transfer_resnet18_report_path = REPORTS_DIR / "transfer_resnet18_pytorch_classification_report.csv"
transfer_resnet18_report_df.to_csv(transfer_resnet18_report_path, encoding="utf-8-sig")

print("Saved:", transfer_resnet18_report_path)
transfer_resnet18_report_df.head(10)

In [ ]:
# Confusion matrix
transfer_resnet18_cm = confusion_matrix(transfer_resnet18_y_true, transfer_resnet18_y_pred)

fig, ax = plt.subplots(figsize=(18, 18))
disp = ConfusionMatrixDisplay(
    confusion_matrix=transfer_resnet18_cm,
    display_labels=class_names
)
disp.plot(ax=ax, xticks_rotation=90, colorbar=False)
plt.title("Transfer ResNet18 PyTorch Confusion Matrix")
plt.tight_layout()

transfer_resnet18_confusion_matrix_path = FIGURES_DIR / "transfer_resnet18_pytorch_confusion_matrix.png"
plt.savefig(transfer_resnet18_confusion_matrix_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", transfer_resnet18_confusion_matrix_path)

In [ ]:
# Save summary
transfer_resnet18_summary = {
    "model_name": "transfer_resnet18_pytorch",
    "image_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs_requested": EPOCHS,
    "epochs_completed": len(transfer_resnet18_history["train_loss"]),
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "num_classes": num_classes,
    "best_val_accuracy": float(max(transfer_resnet18_history["val_acc"])),
    "best_val_loss": float(min(transfer_resnet18_history["val_loss"])),
    "test_loss": float(transfer_resnet18_test_loss),
    "test_accuracy": float(transfer_resnet18_test_acc),
    "freeze_backbone": FREEZE_BACKBONE,
    "device": str(device)
}

transfer_resnet18_summary_path = REPORTS_DIR / "transfer_resnet18_pytorch_summary.json"
with open(transfer_resnet18_summary_path, "w", encoding="utf-8") as f:
    json.dump(transfer_resnet18_summary, f, ensure_ascii=False, indent=4)

print("Saved:", transfer_resnet18_summary_path)
transfer_resnet18_summary

## Notes for report writing

This notebook can be described as a **transfer learning model based on pretrained ResNet18**.

Main characteristics:
- pretrained on ImageNet
- final fully connected layer replaced for 50-class classification
- trained on GPU
- validation-based checkpointing
- data augmentation and normalization
- suitable for comparison against custom CNN models

This model serves as the **second major model family** in the project.
